# Unidad 7 · Clase 2 — Clasificación con k-NN

En la Clase 1 predijimos **cuánto vale** un diamante (regresión). Hoy usamos los mismos diamantes y la misma X, pero predecimos **a qué categoría de precio pertenece** (clasificación).

1. De regresión a clasificación
2. k-NN, accuracy y matriz de confusión
3. Validación cruzada y GridSearch
4. Escalado y Pipeline
5. Práctica de regresión: consumo de autos (`mpg`)
6. Práctica de clasificación: diagnóstico de tumores (`breast_cancer`)

# 0. Importar librerías

In [ ]:
# Librerías para EDA
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Librerías para Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_wine, load_breast_cancer

# Métricas de clasificación
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
# Métricas de regresión (para la práctica 5)
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. De regresión a clasificación

| | Clase 1 · Regresión | Hoy · Clasificación |
|---|---|---|
| Pregunta | ¿Cuánto vale? | ¿De qué categoría es? |
| X | quilates | quilates |
| y | precio (número) | categoria_precio (categoría) |
| Modelo | `KNeighborsRegressor` | `KNeighborsClassifier` |
| Métricas | RMSE, MAE, R² | Accuracy, matriz de confusión |

**La tarea la define el tipo de y**, no el algoritmo: k-NN sirve para las dos.

In [ ]:
# Cargar dataset
df_raw = pd.read_csv("https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/diamonds_clean.csv")
df_raw.shape


In [ ]:
# Seleccionar columnas de interés y renombrarlas 
df = df_raw[["carat", "price", "categoria_precio"]].rename(columns={
    "price":"precio",
    "carat": "quilates",
    "categoria_precio": "categoria"})

In [ ]:
# Inspeccionamos la variable objetivo (variable a predecir)
df["categoria"].value_counts()

In [ ]:
SEMILLA = 42
N_SAMPLE = 5000 # Máximo 53794

En el CSV la categoría está escrita **"Premiun"** (con n). La corregimos para que las tablas y los gráficos queden bien.

In [ ]:
df["categoria"] = df["categoria"].replace({"Premiun": "Premium","Económico": "Economico"})

# ¿Cómo se definió la categoría? Mínimo y máximo de precio en cada una
df.groupby("categoria")["precio"].agg(["min", "max", "count"])
# df.groupby("categoria")["precio"].describe()

La categoría es el precio cortado en tres rangos: **menos de 1.000 USD**, **de 1.000 a 4.999** y **5.000 o más**.

> ⚠️ Por eso **el precio no puede ir en X**: la categoría se calcula a partir de él. Sería como darle al modelo la respuesta (fuga de datos).

In [ ]:
# EDA: precio vs quilates, coloreado por categoría (muestra de N_SAMPLE diamantes)
muestra = df.sample(N_SAMPLE, random_state=SEMILLA)
orden = ["Economico", "Medio", "Premium"]

fig, ax = plt.subplots(figsize=(11, 5))
sns.scatterplot(data=muestra, x="quilates", y="precio", hue="categoria",
                hue_order=orden, palette=["#8FA3BF", "#14213D", "#C9661A"], s=12, alpha=0.5, ax=ax)
ax.set_xlim(0.15, 2.6)
ax.set_title("Cada categoría ocupa una franja de quilates")
plt.tight_layout()
plt.show()

# 2. k-NN, accuracy y matriz de confusión

## 2.1 Separar train y test con `stratify`

En clasificación agregamos `stratify=y`: así train y test tienen **la misma proporción de cada categoría** que el dataset completo.

In [ ]:
# Variables independientes o predictoras X
X = df[["quilates"]]

# Variable objetivo o variable respuesta y
y = df["categoria"]

In [ ]:
# Proporción de cada categoría: las clases NO están balanceadas
y.value_counts(normalize=True).loc[orden].round(3)

In [ ]:
# Split manual - elegimos un 80/20 pero con stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA, stratify=y) # Quitar stratify y ver que cambian

# Exploramos como quedo dividido el conjunto de train y test
print("Train:", X_train.shape, " Test:", X_test.shape)
pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).loc[orden].round(3)

## 2.2 El baseline: predecir siempre la clase más frecuente

En regresión el baseline era predecir siempre la media. En clasificación es predecir siempre **la categoría más frecuente**. `DummyClassifier` hace exactamente eso.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

acc_baseline = accuracy_score(y_test, baseline.predict(X_test))
print(f"Baseline (siempre '{baseline.predict(X_test)[0]}'): {acc_baseline:.1%}")

## 2.3 k-NN clasificador: en vez de promediar, vota

Mismo flujo que en la Clase 1: crear, `fit`, `predict`. Lo único que cambia es la clase: `KNeighborsClassifier`. Para cada diamante busca los 5 más parecidos y devuelve **la categoría más votada**.

[Playground vizlearn - KNN Classificator](https://vizlearn.app/ai/classical-ml/k-nearest-neighbors)


In [ ]:
# Crear el modelo
kmodel_nn = KNeighborsClassifier(n_neighbors=5)

# Entrenamos el modelo a partir de los datos de training
kmodel_nn.fit(X_train, y_train)

# Usamos el modelo resultante para predecir las clases (en regresión eran precios) a partir de los quilates del conjunto de testing (X_test)
y_pred = kmodel_nn.predict(X_test)

y_pred[:10]

In [ ]:
# Un diamante de test y sus 5 vecinos: ¿cómo votaron?
i = 0
diamante = X_test.iloc[[i]]
distancias, posiciones = kmodel_nn.kneighbors(diamante)

vecinos = X_train.iloc[posiciones[0]].copy()
vecinos["categoria"] = y_train.iloc[posiciones[0]].values
print(f"Diamante de {diamante['quilates'].iloc[0]} quilates · real: {y_test.iloc[i]} · predicción: {y_pred[i]}")
vecinos

## 2.4 Accuracy: el porcentaje de aciertos

La métrica Accuracy permite evaluar el desempeño general de nuestro modelo KNN, indicando qué proporción de los diamantes fue clasificada correctamente respecto del total de predicciones realizadas. Su valor se encuentra entre 0 y 1, donde 1 representa una clasificación completamente correcta. Si bien es una métrica sencilla de interpretar, puede resultar engañosa cuando las categorías presentan cantidades muy diferentes de observaciones (clases desbalanceadas).

In [ ]:
acc_knn = accuracy_score(y_test, y_pred)

print("Accuracy/Exactitud:")
print(f"Baseline: {acc_baseline:.1%}")
print(f"k-NN k=5: {acc_knn:.1%}")
print(f"\nAciertos: {(y_test == y_pred).sum():,} de {len(y_test):,}")

## 2.5 Matríz de confusión

La Matriz de Confusión nos permite evaluar cómo nuestro modelo KNN clasificó los diamantes, comparando las clases reales con las clases predichas. 

A diferencia de la métrica Accuracy, que indica el porcentaje global de aciertos, esta matriz permite identificar cuántos diamantes fueron clasificados correctamente en cada categoría y cuáles fueron confundidos con otras clases. De esta manera, podemos analizar con mayor detalle los aciertos y errores del modelo y detectar en qué categorías presenta mayores dificultades.

[Plyground vizlearn - Confusion Matrix](https://vizlearn.in/machine_learning/confusion_matrix.html)

In [ ]:
# Calcular la matríz de confusión
cm = confusion_matrix(y_test, y_pred, labels=orden)
pd.DataFrame(cm, index=[f"real {c}" for c in orden], columns=[f"pred {c}" for c in orden])

In [ ]:
# Graficamos la matríz de confusión
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=orden).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("k-NN (k=5): matriz de confusión")
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

**Cómo leerla:**

- **Nunca confunde los extremos**: ningún Económico se predice como Premium, ni al revés. Los errores son siempre con la categoría vecina.
- **El error más común** es predecir Premium para un diamante Medio. ¿Qué diamantes son?

In [ ]:
# Los diamantes Medios que el modelo predijo como Premium
error = (y_test == "Medio") & (y_pred == "Premium")
print("Cantidad:", error.sum())
X_test[error]["quilates"].describe().round(2)

Son todos diamantes de **1,01 quilates o más** que cuestan menos de 5.000 USD. Con solo los quilates, el modelo no tiene forma de saber que son baratos para su tamaño (el corte, el color o la claridad lo explicarían).

## 2.6 Evaluación del modelo con Precision, Recall y F1-Score

Además de la matriz de confusión y la métrica Accuracy, podemos utilizar Precision, Recall y F1-Score para evaluar con mayor detalle el desempeño de nuestro modelo KNN en la clasificación de diamantes. 

[Playground vizlearn - Precision, Recall and F1](https://vizlearn.in/machine_learning/precision_recall_and_f1.html)



In [ ]:
print(classification_report(y_test, y_pred, labels=orden, digits=3))

`accuracy`: aciertos / total de casos

`precision`: de los que el modelo predijo en esa clase, ¿cuántos eran de verdad? TP / (TP + FP) Mirar verticalmente

`recall`: de los que eran de esa clase, ¿cuántos encontró? TP / (TP + FN) Mirar horizontalmente

`f1-score`: el promedio armónico de precision y recall. Resume las dos en un número y baja si alguna de las dos es mala.

### Conclusión: accuracy y F1 en los diamantes

**Las dos métricas cuentan casi la misma historia.** El accuracy (0,838) y el F1 macro (0,843) dan prácticamente lo mismo. Pasa porque las clases están razonablemente balanceadas (27% · 46% · 27%) y porque ningún error es más grave que otro: confundir un Medio con un Premium cuesta lo mismo que confundir un Económico con un Medio. En un problema así, el accuracy alcanza para comparar modelos, y es la más fácil de explicar.

**Donde se separan es frente al baseline.** Un modelo que predice siempre «Medio» tiene 45,8% de accuracy, que no parece tan malo. Su F1 macro, en cambio, es de apenas 0,21: el F1 de Económico y de Premium es 0, porque nunca los predice. El F1 macro castiga a un modelo que ignora clases; el accuracy no.

**El F1 por clase muestra dónde le cuesta al modelo:**

- **Económico** tiene el F1 más alto (0,871): es la clase más fácil, porque los diamantes chicos casi siempre son baratos.
- **Medio** tiene el más bajo (0,821): es la clase del medio y se confunde para los dos lados. Se le escapan 307 hacia Económico y 643 hacia Premium.
- **Premium** tiene recall alto (0,876) pero precision más baja (0,800). El modelo encuentra casi todos los Premium, pero a cambio marca como Premium a 643 diamantes Medios de entre 1,01 y 1,52 quilates.

**Para mejorar no alcanza con cambiar k:** el error está en los diamantes del mismo tamaño con precios distintos. Hacen falta más variables, como el corte, el color o la claridad.

# 3. Validación cruzada y GridSearch

Igual que en la Clase 1: **k no se elige mirando el test**. Usamos validación cruzada dentro de train.

En clasificación se usa `StratifiedKFold`: cada una de las 5 partes mantiene la proporción de categorías. Y, como el CSV no viene en orden aleatorio, `shuffle=True`.

In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

scores = cross_val_score(KNeighborsClassifier(n_neighbors=5), X_train, y_train,
                         cv=kfold, scoring="accuracy")

print("Accuracy de cada ronda:", scores.round(3))
print(f"Promedio: {scores.mean():.1%} ± {scores.std():.1%}")

En clasificación el scoring es `"accuracy"` y **no lleva signo negativo**: más alto ya es mejor.

In [ ]:
grilla = {"n_neighbors": [1, 5, 15, 50, 100, 200, 400]}

busqueda = GridSearchCV(KNeighborsClassifier(), grilla, cv=kfold, scoring="accuracy")
busqueda.fit(X_train, y_train)   # 7 valores x 5 rondas = 35 modelos

print("Mejor k:", busqueda.best_params_)
print(f"Accuracy de validación: {busqueda.best_score_:.1%}")

In [ ]:
# Curva de validación
res = pd.DataFrame(busqueda.cv_results_)
k_vals = res["param_n_neighbors"].astype(int)
media = res["mean_test_score"] * 100
desvio = res["std_test_score"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(k_vals, media - desvio, media + desvio, color="tab:orange", alpha=0.2, label="± 1 desvío entre folds")
ax.plot(k_vals, media, marker="o", color="tab:orange", label="Accuracy promedio de validación")
ax.axhline(acc_baseline * 100, color="gray", linestyle=":", label=f"Baseline ({acc_baseline:.1%})")
ax.set_xscale("log")
ax.set_xticks(k_vals)
ax.set_xticklabels(k_vals)
ax.set_xlabel("k (escala log)")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Curva de validación de k-NN clasificador")
ax.legend()
plt.tight_layout()
plt.show()

- Con **k = 1** el accuracy es el más bajo y el más variable entre rondas (sobreajuste).
- **Desde k = 50 la curva casi no sube**: menos de medio punto. Ante un empate práctico, se puede elegir un k más chico y rápido.

Ahora sí, **una sola vez**, el test:

In [ ]:
mejor_knn = busqueda.best_estimator_      # ya reentrenado con todo train
y_pred_cv = mejor_knn.predict(X_test)

print(f"Accuracy en test (k={busqueda.best_params_['n_neighbors']}): {accuracy_score(y_test, y_pred_cv):.1%}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_cv, labels=orden, ax=ax, cmap="Blues", colorbar=False)
ax.set_title("k-NN elegido con validación cruzada")
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

# 4. Escalado y Pipeline 
**Análisis con [Dataset Wines](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html) de Scikit-learn**

Analizar el [Playground de VizLearn](https://vizlearn.in/machine_learning/feature_scaling.html)

Con los diamantes había **una sola columna** en X (quilates), así que escalar no cambiaba nada. 
Para ver por qué importa, usamos un dataset con **muchas columnas en escalas muy distintas**: los vinos de scikit-learn (178 vinos, 13 variables químicas, 3 variedades).

In [ ]:
# Cargar el dataset wine
vinos = load_wine(as_frame=True)

# el parámetro as_frame indica que utilice estructuras de Pandas

In [ ]:
# Extraer las variables predictoras y las clases
X_v = vinos.data
y_v = vinos.target

In [ ]:
# EDA reducido - Ver U7_NB4_vinos para un EDA más completo
# X_v.info()

In [ ]:
# Rango de cada columna: algunas van de 0 a 1, otra de 278 a 1.680
X_v.describe().loc[["min", "max"]].T.sort_values("max")

In [ ]:
# Clases en y_v
y_v.value_counts()

In [ ]:
# Split en train/test
Xv_train, Xv_test, yv_train, yv_test = train_test_split(
    X_v, y_v, test_size=0.2, random_state=SEMILLA, stratify=y_v)

In [ ]:
# k-NN sin escalar
knn_crudo = KNeighborsClassifier(n_neighbors=5).fit(Xv_train, yv_train)
yv_predict = knn_crudo.predict(Xv_test)
accuracy = accuracy_score(yv_test, yv_predict)
print(f"k-NN sin escalar: {accuracy:.1%}")

**El problema:** k-NN mide distancias. `proline` (de 278 a 1.680) domina la distancia y las otras 12 columnas casi no cuentan.

**La solución:** `StandardScaler` lleva cada columna a media 0 y desvío 1. Y para no olvidarnos de escalar test con la media de train, lo encadenamos con el modelo en un **Pipeline**:

- `pipe.fit(X_train, y_train)` → el scaler calcula media y desvío **solo con train**, transforma y entrena.
- `pipe.predict(X_test)` → transforma test **con esa misma** media y desvío, y predice.

In [ ]:
# Armar el Scikit-learn pipeline con StandarScaler y KNN Classifier
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])


In [ ]:
# Finalmente entrenamos y evaluamos la métrica Accuracy
pipe.fit(Xv_train, yv_train)
print(f"k-NN escalado: {accuracy_score(yv_test, pipe.predict(Xv_test)):.1%}")

In [ ]:
# Aplicar el scaler entrenado dentro del pipeline
X_train_scaled = pd.DataFrame(
    pipe.named_steps["scaler"].transform(Xv_train),
    columns=Xv_train.columns,
    index=Xv_train.index
)

# Mostrar mínimos y máximos de las variables escaladas
X_train_scaled.describe().loc[["min", "max"]].T.sort_values("max")

- **Diamantes**: con una sola columna, escalar no cambia nada (por eso la fila 2 y la 3 son iguales). El techo de ~85% lo pone la falta de variables, no el algoritmo.
- **Vinos**: escalar cambia todo. Con 13 columnas en escalas distintas, k-NN necesita el Pipeline.

# 5. Práctica de regresión: consumo de autos

Dataset `mpg` de seaborn: 398 autos de los años 70 y 80. Queremos predecir el **consumo en millas por galón** (`mpg`: más millas, menos consumo) a partir del peso, la potencia y la aceleración.

**Consignas**

1. Cargá el dataset y revisá la **calidad**: ¿hay nulos? ¿en qué columna?
2. Explorá la relación de `mpg` con `weight`, `horsepower` y `acceleration` (scatterplots y correlación).
3. Separá 80/20 con `random_state=42`.
4. Entrená una **regresión lineal** y un **k-NN** (k=5). Calculá RMSE y R², y compará con el baseline (predecir la media).
5. Probá k-NN **con Pipeline y escalado**, y elegí k con `GridSearchCV`. ¿Mejora? ¿Por qué?

---
### Solución

In [ ]:
mpg = sns.load_dataset("mpg")
print(mpg.shape)
mpg.isna().sum()

In [ ]:
# 6 autos sin potencia: los sacamos (scikit-learn no acepta nulos)
mpg = mpg.dropna(subset=["horsepower"])
print(mpg.shape)

In [ ]:
variables = ["weight", "horsepower", "acceleration"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, col in zip(axes, variables):
    sns.scatterplot(data=mpg, x=col, y="mpg", ax=ax, alpha=0.6)
plt.tight_layout()
plt.show()

mpg[variables + ["mpg"]].corr()["mpg"].round(2)

In [ ]:
Xm = mpg[variables]
ym = mpg["mpg"]
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.2, random_state=42)

def evaluar_reg(nombre, y_real, y_pred):
    return {"modelo": nombre,
            "RMSE": root_mean_squared_error(y_real, y_pred),
            "MAE": mean_absolute_error(y_real, y_pred),
            "R²": r2_score(y_real, y_pred)}

rl = LinearRegression().fit(Xm_train, ym_train)
knn_m = KNeighborsRegressor(n_neighbors=5).fit(Xm_train, ym_train)

resultados_mpg = [
    evaluar_reg("Media (baseline)", ym_test, np.full(len(ym_test), ym_train.mean())),
    evaluar_reg("Regresión lineal", ym_test, rl.predict(Xm_test)),
    evaluar_reg("k-NN k=5 sin escalar", ym_test, knn_m.predict(Xm_test)),
]
pd.DataFrame(resultados_mpg).round(2)

In [ ]:
pipe_m = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())])
busqueda_m = GridSearchCV(pipe_m, {"knn__n_neighbors": [1, 3, 5, 7, 10, 15, 20, 30, 50]},
                          cv=KFold(n_splits=5, shuffle=True, random_state=42),
                          scoring="neg_root_mean_squared_error")
busqueda_m.fit(Xm_train, ym_train)
print("Mejor:", busqueda_m.best_params_)

resultados_mpg.append(evaluar_reg(f"k-NN escalado (k={busqueda_m.best_params_['knn__n_neighbors']})", ym_test, busqueda_m.predict(Xm_test)))
pd.DataFrame(resultados_mpg).round(2)

**Para discutir:** acá escalar **no mejora**. Sin escalar, `weight` (de 1.600 a 5.100 libras) domina la distancia... pero `weight` es justamente la variable más relacionada con el consumo (correlación −0,83). La escala "eligió" bien por casualidad. Con 79 autos de test, las diferencias entre modelos son chicas y ruidosas.

# 6. Práctica de clasificación: diagnóstico de tumores

Dataset `load_breast_cancer` de scikit-learn: 569 tumores con 30 medidas de las células. Queremos predecir si el tumor es **maligno o benigno**.

**Consignas**

1. Cargá el dataset con `as_frame=True`. ¿Cuántos casos de cada clase hay? Ojo: **0 = maligno, 1 = benigno**.
2. Separá 80/20 con `random_state=42` y `stratify=y`.
3. Entrená k-NN (k=5) sin escalar: accuracy y matriz de confusión.
4. Repetí con Pipeline y escalado. Elegí k con `GridSearchCV`.
5. En la matriz de confusión, ¿cuántos tumores **malignos** se predijeron como **benignos**? ¿Por qué ese error es más grave que el contrario?

---
### Solución

In [ ]:
tumores = load_breast_cancer(as_frame=True)
X_t = tumores.data
y_t = tumores.target
print(X_t.shape)
y_t.map({0: "maligno", 1: "benigno"}).value_counts()

In [ ]:
# Escalas muy distintas: de milésimas a miles
X_t.describe().loc[["min", "max"]].T.sort_values("max").iloc[[0, 1, -2, -1]]

In [ ]:
Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t)

nombres = ["maligno", "benigno"]   # 0 y 1

knn_t = KNeighborsClassifier(n_neighbors=5).fit(Xt_train, yt_train)
print(f"Baseline:          {yt_test.value_counts(normalize=True).max():.1%}")
print(f"k-NN sin escalar:  {accuracy_score(yt_test, knn_t.predict(Xt_test)):.1%}")

busqueda_t = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())]),
    {"knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21, 31]},
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring="accuracy")
busqueda_t.fit(Xt_train, yt_train)
print(f"k-NN escalado (k={busqueda_t.best_params_['knn__n_neighbors']}): {accuracy_score(yt_test, busqueda_t.predict(Xt_test)):.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(yt_test, knn_t.predict(Xt_test), display_labels=nombres,
                                        ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Sin escalar (k=5)")
ConfusionMatrixDisplay.from_predictions(yt_test, busqueda_t.predict(Xt_test), display_labels=nombres,
                                        ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_title(f"Escalado (k={busqueda_t.best_params_['knn__n_neighbors']})")
for ax in axes:
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

**Para discutir:** la celda *real maligno → predicho benigno* es un **falso negativo**: un tumor maligno que el modelo deja pasar. En un diagnóstico es mucho más grave que el error contrario (un benigno que se manda a estudiar de más). Por eso, en estos problemas, el accuracy solo no alcanza: hay que mirar **en qué** se equivoca el modelo.